# 30-Day Hospital Readmission Risk Prediction
### Structured Data Cleaning, Preprocessing & EDA Storytelling Pipeline

> **Course:** DS312 — Data Mining and Applications  
> **Instructor:** Nicole S. Menorias  
> **Academic Activity:** Module 4 — Preprocessing for Structured Data | Laboratory 2: Structured Data Cleaning and EDA Storytelling Report  
> **Dataset:** [Diabetes 130-US Hospitals (1999–2008)](https://www.kaggle.com/datasets/anushrevankar/uci-diabetes-130-us-hospitals-dataset)  
> **Repository:** [Troge-dev/Diabetes-130-US-Hospitals-30-Day-Readmission-Prediction](https://github.com/Troge-dev/Diabetes-130-US-Hospitals-30-Day-Readmission-Prediction)

## 1. Project Goal & Core Questions

### Why This Project Matters
When diabetic patients leave the hospital, returning unexpectedly within 30 days is a serious problem. It often means recovery was disrupted, medications need adjustment, or complications arose. For hospitals, high readmission rates also lead to substantial financial penalties under programs like CMS HRRP.

### Project Scope: Preprocessing & EDA Only
> **Important Scope Note:**  
> This notebook focuses **strictly on the data preprocessing stage** — including data cleaning, missing value diagnostics, feature transformation, numerical scaling, categorical encoding, and exploratory data analysis (EDA).  
> **No machine learning models will be trained or built in this notebook.**  
> Instead, our objective is to produce a clean, model-ready dataset and, at the end of preprocessing, use our findings to determine and **recommend what machine learning model is best suited** for predicting 30-day readmissions.

---

### The Main Goal
The primary goal is to take raw, messy electronic health records from 130 US hospitals, diagnose data quality issues, and transform them into a reliable, structured dataset. Through this pipeline, we will uncover key clinical patterns and risk factors associated with **unplanned 30-day readmissions in diabetic patients**.

---

### Questions This Notebook Aims to Answer

1. **Clinical & Risk Pattern Exploration:**  
   *What are the main clinical signs (such as past hospital visits, lab tests, and medication changes) associated with an early 30-day readmission?*

2. **Statistical Hypothesis Question:**  
   *Do patients who stay longer in the hospital during their initial visit have a significantly higher rate of returning within 30 days than patients who stay for shorter periods?*

3. **Practical Risk Indicators:**  
   * **Prior Visits:** Does a history of emergency room or inpatient visits strongly increase a patient's chance of coming back?  
   * **Medication Instability:** Does changing a patient's insulin dosage during their stay indicate blood sugar instability that elevates readmission risk?  

4. **Downstream Modeling Recommendation (End of Preprocessing):**  
   *Based on the resulting data distribution, class balance, outliers, and feature relationships uncovered in this analysis, **which machine learning model is best suited** to solve this prediction problem downstream?*

## 2. Dataset Context, Raw Metadata & Deep Feature Inspection

### 2.1 Clinical Background & Context
This dataset represents **10 years of clinical care (1999–2008)** across **130 hospitals and integrated healthcare networks** throughout the United States. It was extracted from the Health Facts database and published by the Center for Clinical and Translational Research at Virginia Commonwealth University.

#### Inclusion Criteria
To ensure clinical relevance, every encounter in this dataset met five strict medical criteria:
1. **Inpatient Hospitalization:** It represents an actual hospital admission (not an outpatient clinic visit).
2. **Diabetic Encounter:** The patient was diagnosed with diabetes or administered antidiabetic medication during the stay.
3. **Length of Stay:** The hospital stay was between **1 and 14 days**.
4. **Laboratory Testing:** Laboratory tests were performed during the hospitalization.
5. **Medication Administration:** Medications were administered to the patient during the stay.

---

### 2.2 Raw Dataset Profile & Summary Metadata

| Metadata Dimension | Details / Value |
| :--- | :--- |
| **Data Source** | UCI Machine Learning Repository & Kaggle ([Diabetes 130-US Hospitals](https://www.kaggle.com/datasets/anushrevankar/uci-diabetes-130-us-hospitals-dataset)) |
| **Timeframe** | 1999 – 2008 (10-Year Clinical Window) |
| **Clinical Setting** | Inpatient Care across 130 US Hospitals |
| **Total Encounters (Rows)** | `101,766` hospital admissions |
| **Unique Patients** | `71,518` distinct individuals (some have multiple repeat admissions) |
| **Raw Attributes (Columns)** | `50` clinical, demographic, operational, and medication features |
| **Missing Value Sentinel** | Missing data is encoded as question marks (`'?'`) rather than standard blanks |
| **Auxiliary Mapping File** | `IDS_mapping.csv` provides text definitions for coded IDs (admission type, source, discharge disposition) |
| **Target Variable** | `readmitted` (3 classes: `<30` days, `>30` days, `NO`) |

In [1]:
# 2.3 Environment Setup & Raw Data Ingestion
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set reproducibility seed and display configurations
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

# Styling aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'Segoe UI', 'DejaVu Sans', 'Arial'

# Ingest raw dataset with explicit missing-value sentinel handling
data_path = os.path.join('..', 'data', 'diabetic_data.csv')
if not os.path.exists(data_path):
    data_path = os.path.join('data', 'diabetic_data.csv')

# Note: keep_default_na=False ensures clinical 'None' for lab tests is preserved
df_raw = pd.read_csv(
    data_path,
    na_values=['?'],
    keep_default_na=False,
    low_memory=False
)

print("Dataset Successfully Loaded:")
print(f" • Total Encounters (Rows): {df_raw.shape[0]:,}")
print(f" • Total Features (Columns): {df_raw.shape[1]}")
print(f" • Unique Patients: {df_raw['patient_nbr'].nunique():,}")
print(f" • In-Memory Footprint: {df_raw.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

Dataset Successfully Loaded:
 • Total Encounters (Rows): 101,766
 • Total Features (Columns): 50
 • Unique Patients: 71,518
 • In-Memory Footprint: 49.17 MB


In [2]:
# 2.4 Overall Feature Inventory & Data Type Audit
# Build comprehensive data dictionary and structural profile across all 50 attributes
dtype_audit = pd.DataFrame({
    'Feature': df_raw.columns,
    'Data_Type': df_raw.dtypes.values,
    'Non_Null_Count': df_raw.notnull().sum().values,
    'Missing_Count': df_raw.isnull().sum().values,
    'Missing_Pct (%)': (df_raw.isnull().sum().values / len(df_raw)) * 100,
    'Unique_Values': df_raw.nunique().values,
    'Sample_Values': [str(df_raw[col].dropna().unique()[:3].tolist()) for col in df_raw.columns]
})

print("Data Types Breakdown:")
print(dtype_audit['Data_Type'].value_counts())
print(f"\nTotal Features with Missing Values: {(dtype_audit['Missing_Count'] > 0).sum()} out of 50")

# Display the feature audit table
display(dtype_audit)

Data Types Breakdown:
Data_Type
str      37
int64    13
Name: count, dtype: int64

Total Features with Missing Values: 7 out of 50


,Feature,Data_Type,Non_Null_Count,Missing_Count,Missing_Pct (%),Unique_Values,Sample_Values
0,encounter_id,int64,101766,0,0.0000,101766,"[2278392, 149190, 64410]"
1,patient_nbr,int64,101766,0,0.0000,71518,"[8222157, 55629189, 86047875]"
2,race,str,99493,2273,2.2336,5,"['Caucasian', 'AfricanAmerican', 'Other']"
3,gender,str,101766,0,0.0000,3,"['Female', 'Male', 'Unknown/Invalid']"
4,age,str,101766,0,0.0000,10,"['[0-10)', '[10-20)', '[20-30)']"
5,weight,str,3197,98569,96.8585,9,"['[75-100)', '[50-75)', '[0-25)']"
6,admission_type_id,int64,101766,0,0.0000,8,"[6, 1, 2]"
7,discharge_disposition_id,int64,101766,0,0.0000,26,"[25, 1, 3]"
8,admission_source_id,int64,101766,0,0.0000,17,"[1, 7, 2]"
9,time_in_hospital,int64,101766,0,0.0000,14,"[1, 3, 2]"


### 2.5 Feature Deep-Dive: Identifiers & Patient Demographics

#### 1. Patient Identifiers (2 features)
* `encounter_id`: Primary database key unique to each hospital admission.
* `patient_nbr`: Unique patient identifier. Because diabetic patients often have repeat hospital visits, `patient_nbr` repeats across multiple admissions.

#### 2. Patient Demographics (3 features)
* `race`: 5 distinct categories (Caucasian, AfricanAmerican, Hispanic, Asian, Other) plus missing values.
* `gender`: Biological sex (Female, Male), plus invalid/unspecified cases.
* `age`: Grouped into 10-year age brackets (`[0-10)`, `[10-20)`, ..., `[90-100)`).

In [3]:
# Analytical Inspection: Category 1 (Identifiers) & Category 2 (Demographics)
print("=== Category 1: Patient Identifiers Inspection ===")
total_encounters = len(df_raw)
unique_patients = df_raw['patient_nbr'].nunique()
encounter_frequencies = df_raw['patient_nbr'].value_counts()

print(f"Total Hospital Encounters: {total_encounters:,}")
print(f"Unique Individual Patients: {unique_patients:,}")
print(f"Patients with exactly 1 admission: {(encounter_frequencies == 1).sum():,} ({(encounter_frequencies == 1).sum()/unique_patients*100:.2f}%)")
print(f"Patients with multiple admissions (>1): {(encounter_frequencies > 1).sum():,} ({(encounter_frequencies > 1).sum()/unique_patients*100:.2f}%)")
print(f"Maximum admissions recorded for a single patient: {encounter_frequencies.max()} visits")

print("\n=== Category 2: Demographics Inspection ===")
print("• Race Distribution (including missing values):")
print(df_raw['race'].value_counts(dropna=False, normalize=True) * 100)

print("\n• Gender Distribution:")
print(df_raw['gender'].value_counts(dropna=False))

print("\n• Age Bracket Distribution:")
print(df_raw['age'].value_counts().sort_index())

=== Category 1: Patient Identifiers Inspection ===
Total Hospital Encounters: 101,766
Unique Individual Patients: 71,518
Patients with exactly 1 admission: 54,745 (76.55%)
Patients with multiple admissions (>1): 16,773 (23.45%)
Maximum admissions recorded for a single patient: 40 visits

=== Category 2: Demographics Inspection ===
• Race Distribution (including missing values):
race
Caucasian         74.7784
AfricanAmerican   18.8766
NaN                2.2336
Hispanic           2.0017
Other              1.4799
Asian              0.6299
Name: proportion, dtype: float64

• Gender Distribution:
gender
Female             54708
Male               47055
Unknown/Invalid        3
Name: count, dtype: int64

• Age Bracket Distribution:
age
[0-10)        161
[10-20)       691
[20-30)      1657
[30-40)      3775
[40-50)      9685
[50-60)     17256
[60-70)     22483
[70-80)     26068
[80-90)     17197
[90-100)     2793
Name: count, dtype: int64


### 2.6 Feature Deep-Dive: Admission Dynamics & Clinical Healthcare Utilization

#### 3. Hospital Admission & Discharge Details (4 features)
* `admission_type_id`: Type of admission (Emergency, Urgent, Elective, Trauma, etc.).
* `discharge_disposition_id`: Destination/status upon discharge (Home, Transfer, SNF, Hospice, Expired).
* `admission_source_id`: Referral route (ER, Physician referral, Clinic transfer).
* `time_in_hospital`: Total inpatient length of stay (integer days ranging from 1 to 14).

#### 4. Clinical Utilization & Healthcare History (6 features)
* `num_lab_procedures`: Count of laboratory tests performed during the visit.
* `num_procedures`: Count of non-lab medical/surgical procedures performed.
* `num_medications`: Count of distinct medications administered during the stay.
* `number_outpatient`: Number of outpatient clinic visits in the 12 months prior to admission.
* `number_emergency`: Number of emergency room visits in the 12 months prior to admission.
* `number_inpatient`: Number of inpatient hospitalizations in the 12 months prior to admission.

In [4]:
# Analytical Inspection: Category 3 (Admission Dynamics) & Category 4 (Clinical Utilization)
print("=== Category 3: Admission & Discharge Dynamics ===")
print("Top 5 Admission Types (IDs):")
print(df_raw['admission_type_id'].value_counts().head(5))

print("\nTop 5 Discharge Dispositions (IDs):")
print(df_raw['discharge_disposition_id'].value_counts().head(5))

print("\nLength of Stay (`time_in_hospital`) Parametric Summary:")
los_stats = df_raw['time_in_hospital'].describe()
print(f"Mean Stay: {los_stats['mean']:.2f} days | Median Stay: {los_stats['50%']:.1f} days | Range: {int(los_stats['min'])}-{int(los_stats['max'])} days")

print("\n=== Category 4: Clinical Utilization & History Summary Statistics ===")
utilization_cols = [
    'num_lab_procedures', 'num_procedures', 'num_medications',
    'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses'
]

util_summary = df_raw[utilization_cols].describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.99]).T
util_summary['skewness'] = df_raw[utilization_cols].skew()
display(util_summary[['mean', 'std', 'min', '50%', '75%', '90%', 'max', 'skewness']])

=== Category 3: Admission & Discharge Dynamics ===
Top 5 Admission Types (IDs):
admission_type_id
1    53990
3    18869
2    18480
6     5291
5     4785
Name: count, dtype: int64

Top 5 Discharge Dispositions (IDs):
discharge_disposition_id
1     60234
3     13954
6     12902
18     3691
2      2128
Name: count, dtype: int64

Length of Stay (`time_in_hospital`) Parametric Summary:
Mean Stay: 4.40 days | Median Stay: 4.0 days | Range: 1-14 days

=== Category 4: Clinical Utilization & History Summary Statistics ===


,mean,std,min,50%,75%,90%,max,skewness
num_lab_procedures,43.0956,19.6744,1.0000,44.0000,57.0000,67.0000,132.0000,-0.2365
num_procedures,1.3397,1.7058,0.0000,1.0000,2.0000,4.0000,6.0000,1.3164
num_medications,16.0218,8.1276,1.0000,15.0000,20.0000,26.0000,81.0000,1.3267
number_outpatient,0.3694,1.2673,0.0000,0.0000,0.0000,1.0000,42.0000,8.8330
number_emergency,0.1978,0.9305,0.0000,0.0000,0.0000,1.0000,76.0000,22.8556
number_inpatient,0.6356,1.2629,0.0000,0.0000,1.0000,2.0000,21.0000,3.6141
number_diagnoses,7.4226,1.9336,1.0000,8.0000,9.0000,9.0000,16.0000,-0.8767


### 2.7 Feature Deep-Dive: Diagnoses, Clinical Acuity & Lab Tests

#### 5. Diagnoses & Clinical Acuity (5 features)
* `weight`: Patient body weight in pounds (heavily missing, ~96.86% `'?'`).
* `payer_code`: Administrative insurance code (e.g. Medicare, Medicaid, private; ~39.56% `'?'`).
* `medical_specialty`: Specialty of admitting/attending physician (~49.08% `'?'`).
* `diag_1`, `diag_2`, `diag_3`: Primary, secondary, and tertiary ICD-9 diagnostic codes (alphanumeric strings, including diabetes `250.xx`, circulatory `390-459`, and respiratory conditions).
* `number_diagnoses`: Total count of diagnoses recorded for the patient.

#### 6. Inpatient Laboratory Tests (2 features)
* `max_glu_serum`: Blood glucose serum test result (`>200`, `>300`, `Norm`, `None`).
* `A1Cresult`: Glycated hemoglobin (HbA1c) test result (`>7`, `>8`, `Norm`, `None`).
* *Crucial Observation:* In both lab tests, `'None'` indicates that the physician **did not order** the test during the encounter. It is a valid clinical category, not missing data.

In [5]:
# Analytical Inspection: Category 5 (Diagnoses & Acuity) & Category 6 (Lab Tests)
print("=== Category 5: Missingness Audit in Acuity & Administrative Fields ===")
acuity_cols = ['weight', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3']
acuity_missing = pd.DataFrame({
    'Feature': acuity_cols,
    'Missing_Count': [df_raw[col].isnull().sum() for col in acuity_cols],
    'Missing_Pct (%)': [(df_raw[col].isnull().sum() / len(df_raw)) * 100 for col in acuity_cols],
    'Unique_Entries': [df_raw[col].nunique() for col in acuity_cols]
})
display(acuity_missing)

print("\nTop 8 Medical Specialties Recorded:")
print(df_raw['medical_specialty'].value_counts().head(8))

print("\nSample ICD-9 Primary Diagnoses (`diag_1`):")
print(df_raw['diag_1'].value_counts().head(8))

print("\n=== Category 6: Inpatient Glycemic Lab Tests Inspection ===")
print("• max_glu_serum Test Ordering Breakdown:")
print(df_raw['max_glu_serum'].value_counts(dropna=False, normalize=True) * 100)

print("\n• A1Cresult (HbA1c) Test Ordering Breakdown:")
print(df_raw['A1Cresult'].value_counts(dropna=False, normalize=True) * 100)

=== Category 5: Missingness Audit in Acuity & Administrative Fields ===


,Feature,Missing_Count,Missing_Pct (%),Unique_Entries
0,weight,98569,96.8585,9
1,payer_code,40256,39.5574,17
2,medical_specialty,49949,49.0822,72
3,diag_1,21,0.0206,716
4,diag_2,358,0.3518,748
5,diag_3,1423,1.3983,789



Top 8 Medical Specialties Recorded:
medical_specialty
InternalMedicine              14635
Emergency/Trauma               7565
Family/GeneralPractice         7440
Cardiology                     5352
Surgery-General                3099
Nephrology                     1613
Orthopedics                    1400
Orthopedics-Reconstructive     1233
Name: count, dtype: int64

Sample ICD-9 Primary Diagnoses (`diag_1`):
diag_1
428    6862
414    6581
786    4016
410    3614
486    3508
427    2766
491    2275
715    2151
Name: count, dtype: int64

=== Category 6: Inpatient Glycemic Lab Tests Inspection ===
• max_glu_serum Test Ordering Breakdown:
max_glu_serum
None   94.7468
Norm    2.5519
>200    1.4592
>300    1.2421
Name: proportion, dtype: float64

• A1Cresult (HbA1c) Test Ordering Breakdown:
A1Cresult
None   83.2773
>8      8.0734
Norm    4.9034
>7      3.7458
Name: proportion, dtype: float64


### 2.8 Feature Deep-Dive: Medications, Dosage Changes & Readmission Target

#### 7. Antidiabetic Medications & Regimen Adjustments (26 features)
* **24 Specific Antidiabetic Drugs:** Across 24 discrete medications, patient dosage status is logged as `No` (not prescribed), `Steady` (maintained dosage), `Up` (dosage titrated upward), or `Down` (dosage titrated downward).
* `change`: Binary indicator of whether any diabetic medication was altered during hospitalization (`Ch` vs `No`).
* `diabetesMed`: Binary indicator of whether any diabetic medication was prescribed (`Yes` vs `No`).

#### 8. Outcome / Target Variable (1 feature)
* `readmitted`: 
  * `<30`: Patient readmitted within 30 days of discharge (High Risk).
  * `>30`: Patient readmitted after more than 30 days.
  * `NO`: No readmission recorded.

In [6]:
# Analytical Inspection: Category 7 (Medications) & Category 8 (Target Variable)
print("=== Category 7: 24 Antidiabetic Medications Audit ===")
med_cols = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
    'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
    'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
    'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone'
]

# Identify zero-variance medications (constants that provide no predictive value)
zero_variance_meds = [col for col in med_cols if df_raw[col].nunique() == 1]
active_meds = [col for col in med_cols if df_raw[col].nunique() > 1]

print(f"Total Medications Evaluated: {len(med_cols)}")
print(f"Zero-Variance (Constant 'No') Medications: {zero_variance_meds}")
print(f"Active Medications with Prescriptions: {len(active_meds)}")

print("\nTop 5 Prescribed Medications (Encounters with active prescription):")
prescribed_counts = {col: (df_raw[col] != 'No').sum() for col in active_meds}
top_prescribed = pd.Series(prescribed_counts).sort_values(ascending=False).head(5)
for med, count in top_prescribed.items():
    print(f" • {med}: {count:,} encounters ({count/len(df_raw)*100:.2f}%)")

print("\nInsulin Titration Profile:")
print(df_raw['insulin'].value_counts(normalize=True) * 100)

print("\nMedication Change Status (`change`):")
print(df_raw['change'].value_counts(normalize=True) * 100)

print("\n=== Category 8: Target Variable (`readmitted`) Distribution ===")
target_counts = df_raw['readmitted'].value_counts()
target_pcts = df_raw['readmitted'].value_counts(normalize=True) * 100
target_summary = pd.DataFrame({
    'Readmission_Class': target_counts.index,
    'Encounter_Count': target_counts.values,
    'Percentage (%)': target_pcts.values
})
display(target_summary)

=== Category 7: 24 Antidiabetic Medications Audit ===
Total Medications Evaluated: 23
Zero-Variance (Constant 'No') Medications: ['examide', 'citoglipton']
Active Medications with Prescriptions: 21

Top 5 Prescribed Medications (Encounters with active prescription):
 • insulin: 54,383 encounters (53.44%)
 • metformin: 19,988 encounters (19.64%)
 • glipizide: 12,686 encounters (12.47%)
 • glyburide: 10,650 encounters (10.47%)
 • pioglitazone: 7,328 encounters (7.20%)

Insulin Titration Profile:
insulin
No       46.5607
Steady   30.3137
Down     12.0060
Up       11.1196
Name: proportion, dtype: float64

Medication Change Status (`change`):
change
No   53.8048
Ch   46.1952
Name: proportion, dtype: float64

=== Category 8: Target Variable (`readmitted`) Distribution ===


,Readmission_Class,Encounter_Count,Percentage (%)
0,NO,54864,53.9119
1,>30,35545,34.9282
2,<30,11357,11.1599


### 2.9 Summary of Key Data Inspection Insights

Our deep inspection of the raw dataset reveals seven empirical findings that directly govern our upcoming preprocessing strategy:

1. **Patient Longitudinal Clustering:** While there are 101,766 hospital encounters, there are only 71,518 unique patients. 16,773 patients have multiple admissions (up to 40 visits). Downstream preprocessing must account for repeated-measures autocorrelation.
2. **Extreme Missingness in Administrative Fields:** `weight` is missing in **96.86%** of encounters, and `payer_code` is missing in **39.56%**. Imputing weight is mathematically indefensible, and payer codes carry no direct biological connection to readmission risk.
3. **Structured Missingness in Specialty:** `medical_specialty` is missing in **49.08%** of records, heavily influenced by whether the admission was an emergency intake or a scheduled referral (MAR mechanism).
4. **Clinical Lab Test Non-Ordering vs Missing Data:** `max_glu_serum` (94.75% `'None'`) and `A1Cresult` (83.28% `'None'`) are **not** missing data; they capture a clinical decision by the attending physician not to order glycemic lab panels.
5. **Zero-Variance Features:** Medications `examide` and `citoglipton` have 100% `'No'` across all 101,766 encounters. They carry zero variance and provide zero predictive utility.
6. **High Skewness in Prior Utilization:** Prior emergency visits (`number_emergency`, skewness = 22.86) and outpatient visits (`number_outpatient`, skewness = 8.83) exhibit extreme positive skew and heavy outliers, requiring robust scaling (e.g. `RobustScaler`).
7. **Severe Class Imbalance:** High-risk 30-day readmission (`<30`) occurs in only **11.16%** of encounters, establishing an imbalanced binary classification setting.